# Mortality component attribution

Native analysis for Figures 1 and 4. Each of the ten hybrid-model members is evaluated under observed weather and the original temperature bounds, holding relative humidity constant where temperature changes. Reference mortality uses 16 °C and 50% relative humidity.

[Original analysis notebook](https://github.com/ClimSocAna/sari-weather-mortality/blob/4402a80d3e490ad7964a3be12597043761bebda1/notebooks/01_mortality_component_attribution.ipynb).

In [ ]:
from pathlib import Path
import sys

repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "configs" / "training.example.yaml").is_file())
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import pandas as pd
from notebooks.utils import AGE_LABELS, daily_index, ensemble, weekly_sum

from notebooks.utils import ANALYSIS_PERIOD, DEVICE, checkpoint_paths
from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor


## Load models for inspection

Run the next cell to load member `000` for each setup into memory. Inspect `model` for the hybrid predictor or `models[setup]` for another predictor. Set `inspection_member` to a value from 0 through 9 to inspect another member. The analysis cells evaluate all ten members in a separate inference workspace.

In [ ]:
inspection_member = 0
setups = ['default']
checkpoints = {setup: checkpoint_paths(setup) for setup in setups}
models = {
    setup: restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
    for setup, paths in checkpoints.items()
}
model = models["default"]
model

## Evaluate temperature counterfactuals

In [ ]:
scenarios = {
    "heat_20": {"maximum_c": 20},
    "cold_15": {"minimum_c": 15},
    "band_15_20": {"minimum_c": 15, "maximum_c": 20},
    "cold_0": {"minimum_c": 0},
}

def components(model, result):
    pred = result.predictions
    factual = pred.total_deaths.sum(dim=1)
    sari = pred.sari_deaths.sum(dim=1)
    covid = pred.covid_deaths.sum(dim=1)
    reference = result.scenarios["reference"].sum(dim=1)
    return {
        "Observed": model.labels.daily_land.sum(dim=1),
        "Predicted": factual,
        "Reference mortality": reference - sari - covid,
        "Non-COVID SARI": sari,
        "COVID-19 SARI": covid,
        "Weather": factual - reference,
        "Heat above 20 °C": factual - result.scenarios["heat_20"].sum(dim=1),
        "Cold below 15 °C": factual - result.scenarios["cold_15"].sum(dim=1),
        "Cold below 0 °C": factual - result.scenarios["cold_0"].sum(dim=1),
        "Outside 15–20 °C": factual - result.scenarios["band_15_20"].sum(dim=1),
        "Population": model.data.population.sum(dim=1),
    }

paths = checkpoints["default"]
inputs = load_run_config(paths[0].parents[1] / "config.yaml").data_config
inference_model = InferenceModel.from_checkpoint(
    paths[0], data_config=inputs, period=ANALYSIS_PERIOD, device=DEVICE,
)
estimates = ensemble(inference_model, paths, components, scenarios=scenarios)
dates = daily_index()

## Figure 1 · Weekly attribution and surveillance

In [ ]:
weeks = dates[::7]
totals = pd.DataFrame({
    name: weekly_sum(value if value.ndim == 1 else value.sum(dim=(1, 2))).numpy()
    for name, value in estimates.items() if name != "Population"
}, index=weeks)
source, = (repo / "data" / "sari").glob("*.tsv")
raw = pd.read_csv(source, sep="\t").query('Altersgruppe == "00+"').copy()
raw["Date"] = pd.to_datetime(raw["Kalenderwoche"] + "-1", format="%G-W%V-%u")
incidence = raw.pivot(index="Date", columns="SARI", values="Hospitalisierungsinzidenz")
incidence = incidence.reindex(weeks)
covid = incidence["COVID-19"].fillna(0)
surveillance = pd.DataFrame({
    "Non-COVID SARI": incidence["Gesamt"] - covid,
    "COVID-19 SARI": covid,
    "Influenza": incidence["Influenza"],
})
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
totals[["Observed", "Predicted", "Reference mortality", "Weather",
        "Non-COVID SARI", "COVID-19 SARI"]].plot(ax=axes[0], ylabel="Deaths per week")
surveillance.plot(ax=axes[1], ylabel="Hospital admissions per 100,000")
totals[["Heat above 20 °C", "Cold below 15 °C", "Cold below 0 °C"]].plot(
    ax=axes[2], ylabel="Associated deaths per week"
)
fig.tight_layout()

## Figure 4 · Age and sex, 2020–2025

In [ ]:
mask = (dates.year >= 2020) & (dates.year <= 2025)
n_years = dates[mask].year.nunique()
population = estimates["Population"][mask].mean(dim=0)
names = ["Non-COVID SARI", "COVID-19 SARI", "Heat above 20 °C", "Cold below 0 °C"]
for sex, index in [("Men", 0), ("Women", 1)]:
    counts = pd.DataFrame({
        name: (estimates[name][mask, :, index].sum(dim=0) / n_years).numpy()
        for name in names
    }, index=AGE_LABELS)
    rates = counts.div(population[:, index].numpy(), axis=0) * 100_000
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    counts.plot.bar(ax=axes[0], title=sex, ylabel="Mean annual deaths")
    rates.plot.bar(ax=axes[1], title=sex, ylabel="Annual deaths per 100,000")
    fig.tight_layout()